### **Notebook 1: Análisis inicial de datos**
##### **Objetivo:** Validar la integridad estructural, codificación de caracteres y consistencia de columnas de los archivos GRD públicos de los años 2019 hasta 2024, con el fin de asegurar la correcta lectura, procesamiento y posterior análisis de pacientes oncológicos.

##### **Procedimiento:** Este notebook realiza una auditoría técnica de los datos fuente GRD mediante:
- Identificación de la codificación de caracteres utilizada en cada archivo .txt.
- Verificación de la correcta lectura de caracteres especiales y acentuados.
- Detección de posibles incompatibilidades entre codificaciones.
- Validación de la estructura tabular de los archivos.
- Detección de filas con número incorrecto de columnas.
- Comparación de esquemas entre años para identificar cambios en variables.
- Generación de evidencia que permita justificar las decisiones de preprocesamiento utilizadas en la construcción de las cohortes oncológicas y control.

##### **1. Detección automática de la codificación de cada archivo (con `chardet`**)

In [ ]:
import os  # Utilidades para manejo de rutas y archivos.
import chardet  # Librería para detectar automáticamente la codificación de texto.
ruta_originales = '../../Datos/Datos originales/'  # Ruta base donde están los archivos originales.
# Diccionario con los datasets por año y el campo clave correspondiente.
datasets = {
    "2019": ("GRD_PUBLICO_2019.txt", "CIP_ENCRIPTADO"),
    "2020": ("GRD_PUBLICO_2020.txt", "CIP_ENCRIPTADO"),
    "2021": ("GRD_PUBLICO_2021.txt", "CIP_ENCRIPTADO"),
    "2022": ("GRD_PUBLICO_2022.txt", "CIP_ENCRIPTADO"),
    "2023": ("GRD_PUBLICO_2023.txt", "CIP_ENCRIPTADO"),
    "2024": ("GRD_PUBLICO_2024.txt", "ID_BENEFICIARIO"),
}
# Iterar sobre cada año y archivo definido en el diccionario.
for año, (archivo, campo) in datasets.items():
    ruta = os.path.join(ruta_originales, archivo)  # Construir la ruta completa del archivo.
    try:
        with open(ruta, "rb") as f:  # Abrir el archivo en modo binario.
            data = f.read()  # Leer todo el contenido del archivo.

        resultado = chardet.detect(data)  # Detectar codificación con chardet.
        cod_detectada = resultado["encoding"]  # Codificación estimada.
        confianza = resultado["confidence"]  # Nivel de confianza en la detección.

        # Mostrar resultados del análisis para cada archivo.
        print(f"Año {año} (archivo {archivo}):")
        print(f"  Tamaño: {len(data):,} bytes")
        print(f"  Codificación detectada:   {cod_detectada} (confianza {confianza:.2f})")
        print()
    except FileNotFoundError:  # Manejo de error si el archivo no existe.
        print(f"Año {año}: Archivo {archivo} no encontrado.\n")

Año 2019 (archivo GRD_PUBLICO_2019.txt):
  Tamaño: 572,538,020 bytes
  Codificación detectada:   utf-8 (confianza 0.99)

Año 2020 (archivo GRD_PUBLICO_2020.txt):
  Tamaño: 397,026,707 bytes
  Codificación detectada:   utf-8 (confianza 0.99)

Año 2021 (archivo GRD_PUBLICO_2021.txt):
  Tamaño: 428,657,825 bytes
  Codificación detectada:   UTF-8-SIG (confianza 1.00)

Año 2022 (archivo GRD_PUBLICO_2022.txt):
  Tamaño: 960,774,530 bytes
  Codificación detectada:   UTF-16 (confianza 1.00)

Año 2023 (archivo GRD_PUBLICO_2023.txt):
  Tamaño: 1,075,450,968 bytes
  Codificación detectada:   UTF-16 (confianza 1.00)

Año 2024 (archivo GRD_PUBLICO_2024.txt):
  Tamaño: 566,268,922 bytes
  Codificación detectada:   ISO-8859-1 (confianza 0.73)



##### **2. Prueba de lectura con distintas codificaciones para el archivo GRD de 2024 (`GRD_PUBLICO_2024.txt`), que tuvo una confianza de codificación inferior a 0.99:** Se intentó probar con las codificaciones `utf-8`, `latin1`, `cp1252`, `utf-16`, `iso-8859-1`.

El archivo 2024 presenta una detección incierta de codificación:  
- Detector automático: ISO‑8859‑1 (confianza 0.73).  
- Lectura inicial: cabeceras legibles en UTF‑8, Latin‑1, cp1252 e ISO‑8859‑1.  
- Lectura en UTF‑16 fallida (no BOM).

In [ ]:
ruta = "../../Datos/Datos originales/GRD_PUBLICO_2024.txt"  # Ruta del archivo 2024 a analizar.

# Lista de codificaciones a probar, incluyendo ISO-8859-1.
codificaciones = ["utf-8", "latin1", "cp1252", "utf-16", "iso-8859-1"]

# Iterar sobre cada codificación y probar lectura.
for enc in codificaciones:
    try:
        with open(ruta, "r", encoding=enc, errors="replace") as f:  # Abrir con la codificación indicada.
            print(f"\n=== {enc} ===")  # Mostrar qué codificación se está probando.
            print(f.readline()[:500])  # Leer y mostrar los primeros 500 caracteres de la primera línea.
    except Exception as e:  # Capturar errores de lectura.
        print(f"{enc}: {e}")


=== utf-8 ===
COD_HOSPITAL|ID_BENEFICIARIO|SEXO|FECHA_NACIMIENTO|ETNIA|PROVINCIA|COMUNA|NACIONALIDAD|PREVISION|SERVICIO_SALUD|TIPO_PROCEDENCIA|TIPO_INGRESO|ESPECIALIDAD_MEDICA|TIPO_ACTIVIDAD|FECHA_INGRESO|SERVICIOINGRESO|FECHATRASLADO1|SERVICIOTRASLADO1|FECHATRASLADO2|SERVICIOTRASLADO2|FECHATRASLADO3|SERVICIOTRASLADO3|FECHATRASLADO4|SERVICIOTRASLADO4|FECHATRASLADO5|SERVICIOTRASLADO5|FECHATRASLADO6|SERVICIOTRASLADO6|FECHATRASLADO7|SERVICIOTRASLADO7|FECHATRASLADO8|SERVICIOTRASLADO8|FECHATRASLADO9|SERVICIOTRASLA

=== latin1 ===
COD_HOSPITAL|ID_BENEFICIARIO|SEXO|FECHA_NACIMIENTO|ETNIA|PROVINCIA|COMUNA|NACIONALIDAD|PREVISION|SERVICIO_SALUD|TIPO_PROCEDENCIA|TIPO_INGRESO|ESPECIALIDAD_MEDICA|TIPO_ACTIVIDAD|FECHA_INGRESO|SERVICIOINGRESO|FECHATRASLADO1|SERVICIOTRASLADO1|FECHATRASLADO2|SERVICIOTRASLADO2|FECHATRASLADO3|SERVICIOTRASLADO3|FECHATRASLADO4|SERVICIOTRASLADO4|FECHATRASLADO5|SERVICIOTRASLADO5|FECHATRASLADO6|SERVICIOTRASLADO6|FECHATRASLADO7|SERVICIOTRASLADO7|FECHATRASLADO8|SERVICIOTRASLAD

##### **3. Conteo de caracteres especiales en distintas codificaciones:** Finalmente, para confirmar la codificación del archivo `GRD_PUBLICO_2024.txt`, se probó un análisis de caracteres especiales (`Ñ`, `ñ`, `Á`, `á`) en distintas codificaciones. Con el fin de confirmar si las tildes y la ñ se preservan correctamente en `latin‑1` o `cp1252`, lo que indicaría que el archivo realmente está en esa codificación.  

- Aunque el detector automático indicó `ISO‑8859‑1` con 73% de confianza para el archivo `GRD_PUBLICO_2024.txt`, las pruebas prácticas de lectura mostraron que tanto `latin1`, `cp1252` como `iso‑8859‑1` interpretan correctamente los caracteres especiales del español (Ñ, Á). Cabe mencionar que, en Python, `latin1` y `iso‑8859‑1` son equivalentes, ya que ambos mapean directamente los bytes 0–255 a los mismos caracteres, por ello, usar `latin1` no implica pérdida de información ni riesgo de errores en este caso.
- Respecto a `latin1`, esta codificación es más común y estándar en scripts de Python, lo que facilita la compatibilidad, ya que la lectura con `latin1` preserva correctamente los caracteres acentuados y la ñ, confirmando que el archivo se interpreta de forma adecuada. En otras palabras, escoger la codificación de `latin1` es una elección práctica y segura, que no va a alterar el contenido real del archivo.
- **Conclusión:** Se recomienda abrir el archivo 2024 con `latin1` para mantener consistencia y simplicidad en el análisis, dado que su comportamiento es idéntico al de `iso‑8859‑1` en este contexto.

In [ ]:
ruta = "../../Datos/Datos originales/GRD_PUBLICO_2024.txt"  # Ruta del archivo 2024.

# Leer un bloque de bytes del archivo (primeros 500.000 bytes).
with open(ruta, "rb") as f:
    data = f.read(500000)

# Probar varias codificaciones, incluyendo iso-8859-1.
for enc in ["latin1", "cp1252", "iso-8859-1"]:
    try:
        texto = data.decode(enc, errors="ignore")  # Decodificar el bloque con la codificación indicada.
        print(f"\n{enc}")  # Mostrar la codificación probada.
        print("Ñ:", texto.count("Ñ"))  # Contar ocurrencias de la letra Ñ mayúscula.
        print("ñ:", texto.count("ñ"))  # Contar ocurrencias de la letra ñ minúscula.
        print("Á:", texto.count("Á"))  # Contar ocurrencias de la letra Á mayúscula.
        print("á:", texto.count("á"))  # Contar ocurrencias de la letra á minúscula.
    except Exception as e:  # Capturar errores de decodificación.
        print(f"{enc}: {e}")



latin1
Ñ: 127
ñ: 0
Á: 134
á: 0

cp1252
Ñ: 127
ñ: 0
Á: 134
á: 0

iso-8859-1
Ñ: 127
ñ: 0
Á: 134
á: 0


##### **4. Validación de estructura de archivos por año:** 

Se revisó cada archivo para confirmar:
- Número de columnas esperadas según la cabecera.
- Total de filas de datos.
- Filas problemáticas con columnas distintas.

**Resultados:**
- **2019:** 129 columnas, 1,151,475 filas, sin problemas.
- **2020:** 129 columnas, 781,912 filas, sin problemas.
- **2021:** 129 columnas, 816,909 filas, sin problemas.
- **2022:** 129 columnas, 932,840 filas, **1 fila problemática** (línea 120127 con 130 columnas).
- **2023:** 129 columnas, 1,039,587 filas, sin problemas.
- **2024:** 129 columnas, 1,085,813 filas, sin problemas.

**Conclusión:** todos los archivos mantienen la misma estructura de 129 columnas, salvo un caso aislado en 2022 que tenía una fila adicional.

In [14]:
import os  # Módulo para manejar rutas y archivos en el sistema.

carpeta = "../../Datos/Datos originales"  # Ruta base donde están los archivos originales.

# Diccionario con los archivos por año y su codificación correspondiente.
archivos = {
    "2019": ("GRD_PUBLICO_2019.txt", "utf-8"),
    "2020": ("GRD_PUBLICO_2020.txt", "utf-8"),
    "2021": ("GRD_PUBLICO_2021.txt", "utf-8-sig"),
    "2022": ("GRD_PUBLICO_2022.txt", "utf-16"),
    "2023": ("GRD_PUBLICO_2023.txt", "utf-16"),
    "2024": ("GRD_PUBLICO_2024.txt", "latin1")
}

# Iterar sobre cada año y archivo definido en el diccionario.
for año, (archivo, encoding) in archivos.items():
    ruta = os.path.join(carpeta, archivo)  # Construir la ruta completa del archivo.
    try:
        with open(ruta, "r", encoding=encoding) as f:  # Abrir el archivo con la codificación indicada.
            encabezado = next(f)  # Leer la primera línea (cabecera).
            columnas_esperadas = len(encabezado.rstrip("\n\r").split("|"))  # Contar columnas esperadas.
            total_filas = 0  # Contador de filas de datos.
            filas_problematicas = 0  # Contador de filas con columnas incorrectas.
            ejemplos = []  # Lista para guardar ejemplos de filas problemáticas.
            # Iterar sobre cada línea de datos a partir de la segunda.
            for num_linea, linea in enumerate(f, start=2):
                total_filas += 1  # Incrementar contador de filas.
                columnas_actuales = len(linea.rstrip("\n\r").split("|"))  # Contar columnas en la fila actual.
                if columnas_actuales != columnas_esperadas:  # Verificar si la fila tiene columnas distintas.
                    filas_problematicas += 1  # Incrementar contador de filas problemáticas.
                    if len(ejemplos) < 10:  # Guardar hasta 10 ejemplos de errores.
                        ejemplos.append((num_linea, columnas_actuales, columnas_esperadas))

        # Mostrar resultados del análisis para el archivo actual.
        print(f"\nAño {año}")
        print(f"Archivo: {archivo}")
        print(f"Columnas esperadas: {columnas_esperadas}")
        print(f"Filas de datos: {total_filas:,}")
        print(f"Filas problemáticas: {filas_problematicas:,}")

        if ejemplos:  # Mostrar ejemplos si existen filas problemáticas.
            print("Primeros ejemplos:")
            for linea, encontradas, esperadas in ejemplos: # Mostrar número de línea, columnas encontradas y columnas esperadas.
                print(f"  Línea {linea}: {encontradas} columnas (esperadas {esperadas})")
        print("-" * 80)  # Separador visual.

    except Exception as e:  # Manejo de errores en la lectura del archivo.
        print(f"\nAño {año}")
        print(f"Error al procesar {archivo}: {e}")
        print("-" * 80)


Año 2019
Archivo: GRD_PUBLICO_2019.txt
Columnas esperadas: 129
Filas de datos: 1,151,475
Filas problemáticas: 0
--------------------------------------------------------------------------------

Año 2020
Archivo: GRD_PUBLICO_2020.txt
Columnas esperadas: 129
Filas de datos: 781,912
Filas problemáticas: 0
--------------------------------------------------------------------------------

Año 2021
Archivo: GRD_PUBLICO_2021.txt
Columnas esperadas: 129
Filas de datos: 816,909
Filas problemáticas: 0
--------------------------------------------------------------------------------

Año 2022
Archivo: GRD_PUBLICO_2022.txt
Columnas esperadas: 129
Filas de datos: 932,840
Filas problemáticas: 1
Primeros ejemplos:
  Línea 120127: 130 columnas (esperadas 129)
--------------------------------------------------------------------------------

Año 2023
Archivo: GRD_PUBLICO_2023.txt
Columnas esperadas: 129
Filas de datos: 1,039,587
Filas problemáticas: 0
-----------------------------------------------------

##### **5. Comparación de columnas entre archivos:**

Se compararon las cabeceras de todos los archivos para detectar diferencias en los nombres de columnas (variables).

**Resultados:**
- Archivos 2019–2023: Correcto, todos tienen las mismas columnas.
- Archivo 2024: Diferencias detectas con el resto de años
  - Los archivos 2019–2023 tienen la columna **CIP_ENCRIPTADO**, ausente en 2024.
  - El archivo 2024 tiene la columna **ID_BENEFICIARIO**, ausente en los anteriores años.

- **Conclusión:** la variable identificadora del paciente en 2024 difiere del resto, sustituyendo el campo `CIP_ENCRIPTADO` por `ID_BENEFICIARIO`.

In [15]:
import os  # Módulo para manejo de rutas y archivos.
# Diccionario para almacenar el conjunto de columnas de cada archivo.
columnas_por_archivo = {}
# Leer únicamente la cabecera de cada archivo para obtener las columnas.
for año, (archivo, encoding) in archivos.items():
    ruta = os.path.join(carpeta, archivo)  # Construir la ruta completa del archivo.
    with open(ruta, "r", encoding=encoding) as f:  # Abrir el archivo con su codificación.
        encabezado = f.readline().strip()  # Leer la primera línea (cabecera).
    columnas = encabezado.split("|")  # Separar las columnas por el delimitador "|".
    columnas_por_archivo[archivo] = set(columnas)  # Guardar columnas como conjunto para comparación.

# Lista de nombres de archivos para realizar comparaciones.
nombres_archivos = list(columnas_por_archivo.keys())

# Análisis comparativo de esquemas entre los archivos.
# Comparar cada par de archivos para detectar diferencias en sus columnas.
for i in range(len(nombres_archivos)):
    for j in range(i + 1, len(nombres_archivos)):
        a1 = nombres_archivos[i]  # Primer archivo del par.
        a2 = nombres_archivos[j]  # Segundo archivo del par.
        cols1 = columnas_por_archivo[a1]  # Conjunto de columnas del primer archivo.
        cols2 = columnas_por_archivo[a2]  # Conjunto de columnas del segundo archivo.
        faltan_en_a1 = cols2 - cols1  # Columnas presentes en a2 pero no en a1.
        faltan_en_a2 = cols1 - cols2  # Columnas presentes en a1 pero no en a2.
        if faltan_en_a1 or faltan_en_a2:  # Si hay diferencias entre los esquemas.
            print(f"\nADVERTENCIA: Diferencias entre {a1} y {a2}")
            if faltan_en_a1:  # Mostrar columnas faltantes en a1.
                print(f"  {a1} NO tiene:")
                for col in sorted(faltan_en_a1):
                    print(f"    - {col}")
            if faltan_en_a2:  # Mostrar columnas faltantes en a2.
                print(f"  {a2} NO tiene:")
                for col in sorted(faltan_en_a2):
                    print(f"    - {col}")
        else:  # Si ambos archivos tienen las mismas columnas.
            print(f"CORRECTO: {a1} y {a2} tienen las mismas columnas")

CORRECTO: GRD_PUBLICO_2019.txt y GRD_PUBLICO_2020.txt tienen las mismas columnas
CORRECTO: GRD_PUBLICO_2019.txt y GRD_PUBLICO_2021.txt tienen las mismas columnas
CORRECTO: GRD_PUBLICO_2019.txt y GRD_PUBLICO_2022.txt tienen las mismas columnas
CORRECTO: GRD_PUBLICO_2019.txt y GRD_PUBLICO_2023.txt tienen las mismas columnas

ADVERTENCIA: Diferencias entre GRD_PUBLICO_2019.txt y GRD_PUBLICO_2024.txt
  GRD_PUBLICO_2019.txt NO tiene:
    - ID_BENEFICIARIO
  GRD_PUBLICO_2024.txt NO tiene:
    - CIP_ENCRIPTADO
CORRECTO: GRD_PUBLICO_2020.txt y GRD_PUBLICO_2021.txt tienen las mismas columnas
CORRECTO: GRD_PUBLICO_2020.txt y GRD_PUBLICO_2022.txt tienen las mismas columnas
CORRECTO: GRD_PUBLICO_2020.txt y GRD_PUBLICO_2023.txt tienen las mismas columnas

ADVERTENCIA: Diferencias entre GRD_PUBLICO_2020.txt y GRD_PUBLICO_2024.txt
  GRD_PUBLICO_2020.txt NO tiene:
    - ID_BENEFICIARIO
  GRD_PUBLICO_2024.txt NO tiene:
    - CIP_ENCRIPTADO
CORRECTO: GRD_PUBLICO_2021.txt y GRD_PUBLICO_2022.txt tienen la